# 02 — Filtros de calidad

Construye una selección limpia a partir del wfset cargado, midiendo por waveform
y por canal cantidades que identifican eventos malos: ruido en baseline, actividad
pre-trigger, amplitud anómala, tiempo de pico desplazado, carga rara y saturación.

El resultado principal es `wfset_quality`. Las razones de rechazo quedan en el
`QualityResult` para poder depurar qué filtro está descartando cada evento.
La lógica vive en `pmtcheck_quality.py`; los valores por defecto en `pmtcheck_config.py`.

| Corte | Descripción | Condición de rechazo | Razón |
|------|-------------|----------------------|--------|
| `baseline_nmad` | Baseline anómala respecto al canal | `abs(baseline - mediana_baseline) > 6 × scale_baseline` | `baseline_shift` |
| `pre_rms_nmad` | Ruido elevado en baseline/pretrigger | `baseline_rms > mediana_baseline_rms + 5 × scale_baseline_rms` | `baseline_rms_high` |
| `pre_max_nmad` | Picos espurios antes del pulso | `pre_max > mediana_pre_max + 6 × scale_pre_max` | `pretrigger_peak` |
| `pre_integral_nmad` | Exceso de carga previa al trigger | `pre_integral_pos > mediana_pre_integral_pos + 6 × scale_pre_integral_pos` | `pretrigger_charge` |
| `peak_amp_nmad` | Amplitud del pulso anómala | `abs(peak_amp - mediana_peak_amp) > 6 × scale_peak_amp` | `peak_amp_outlier` |
| `charge_nmad` | Carga integrada anómala | `abs(charge - mediana_charge) > 6 × scale_charge` | `charge_outlier` |
| `peak_tick_abs` | Desplazamiento temporal del pico | `abs(peak_tick - mediana_peak_tick) > 4` | `peak_time_shift` |
| `peak_amp_abs_min` | Pulso demasiado pequeño | `peak_amp < 400` | `peak_amp_low` |
| `peak_amp_abs_max` | Pulso demasiado grande | `peak_amp > 7000` | `peak_amp_high` |
| `adc_abs_max` | Saturación ADC cruda | `abs(raw_adc_min) > adc_abs_max OR abs(raw_adc_max) > adc_abs_max` | `raw_adc_saturation` |
| `adc_min_threshold` | Saturación negativa del ADC | `raw_adc_min < -1000` | `adc_negative_saturation` |
| `sustained_amp_min` | Amplitud sostenida en la ventana del pulso (antiguo filtro adicional de amplitud) | `min(y[ventana_amp]) <= 400` | `sustained_amp_low` |

Nota: el antiguo "filtro adicional de amplitud" está integrado como `sustained_amp_min`
(ventana 75–80, canal 16: 78–83); su corte superior (`< 7000`) ya lo cubre `peak_amp_abs_max`.

In [ ]:
%load_ext autoreload
%autoreload 2

from waffles.data_classes.ChannelWsGrid import ChannelWsGrid

from pmtcheck_io import load_wfset
from pmtcheck_quality import build_quality_wfset, plot_quality_diagnostics
import pmtcheck_viewers as viewers

In [ ]:
run = 43363
wfset = load_wfset(run)

## Configuración de los cortes

Copia los valores por defecto y ajusta aquí lo que quieras probar.

In [ ]:
from pmtcheck_config import AMP_CHECK_WINDOWS, QUALITY_CUTS, QUALITY_WINDOWS

cuts = dict(QUALITY_CUTS)
windows = dict(QUALITY_WINDOWS)
amp_windows = dict(AMP_CHECK_WINDOWS)

# Ajustes de esta sesion, p. ej.:
# cuts["peak_tick_abs"] = 6
# windows["signal"] = slice(60, 120)

cuts

## Aplicar los filtros

In [ ]:
wfset_quality, qres = build_quality_wfset(wfset, cuts=cuts, windows=windows, amp_windows=amp_windows)

In [ ]:
wfch_quality = ChannelWsGrid.clusterize_waveform_set(wfset_quality)
wfch_quality

## Diagnósticos de los filtros

Métricas disponibles para `metric`: `baseline`, `baseline_rms`, `pre_max`,
`pre_integral_pos`, `peak_tick`, `peak_amp`, `charge`, `raw_adc_min`,
`raw_adc_max`, `amp_window_min`.

In [ ]:
plot_quality_diagnostics(qres, endpoint=110, channel=14, metric="peak_amp")

## Visor interactivo de waveforms por canal después del filtrado

In [ ]:
viewers.browse_channel_waveforms(wfset_quality, endpoint=110)